# 02 · Classical baselines (TF-IDF + LogReg / SVM)

Phase 3. Trains TF-IDF + Logistic Regression and TF-IDF + calibrated linear SVM, tunes per-label thresholds on validation only, evaluates on test with `src/evaluate.py`, and logs every run to MLflow. No GPU needed.

## Setup
Clones the repo, installs packages, and links `models/` and `mlruns/` to Google Drive so nothing is lost if Colab disconnects.

In [ ]:
# 1. Get the code (for a private repo, see the README: add GITHUB_TOKEN in Colab Secrets)
import os
if not os.path.exists("/content/fineguard"):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        !git clone -q https://{token}@github.com/424103-achu/fineguard.git /content/fineguard
    except Exception:
        !git clone -q https://github.com/424103-achu/fineguard.git /content/fineguard
%cd /content/fineguard/ml
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
# 2. Save outputs to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/fineguard"
!mkdir -p {DRIVE}/models {DRIVE}/mlruns {DRIVE}/figures
!rm -rf models mlruns && ln -sfn {DRIVE}/models models && ln -sfn {DRIVE}/mlruns mlruns

# MLflow's SQLite file works best on local disk: copy it from Drive now, back at the end
import os, shutil
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////content/mlflow.db"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
if os.path.exists(f"{DRIVE}/mlruns/mlflow.db"):
    shutil.copy(f"{DRIVE}/mlruns/mlflow.db", "/content/mlflow.db")
FIG = f"{DRIVE}/figures"

## Main runs (word 1–2 grams, balanced class weights)

In [ ]:
!python -m src.train_classical

## Ablations
Each is its own logged MLflow run. The `m-F1 (8) @0.5` column of the results table is the fixed-threshold score, so the gain from threshold tuning is visible for every row.

In [ ]:
!python -m src.train_classical --ngram 1 1 --out-dir models/ablations
!python -m src.train_classical --no-balanced --out-dir models/ablations
!python -m src.train_classical --c-logreg 1 --c-svm 0.1 --models tfidf_logreg tfidf_svm --out-dir models/ablations

## Per-category report and error analysis for the best baseline

In [ ]:
import joblib, numpy as np
from src.data import load_all_splits
from src.evaluate import apply_thresholds, per_label_report
from src.error_analysis import error_table, confusion_pairs, f1_by_length

data, LABELS = load_all_splits()
te_x, te_y = data["test"]
bundle = joblib.load("models/tfidf_svm.joblib")
pred = apply_thresholds(bundle["model"].predict_proba(bundle["vectorizer"].transform(te_x)), bundle["thresholds"])
print(per_label_report(te_y, pred, LABELS))
display(f1_by_length(te_x, te_y, pred))
display(confusion_pairs(te_y, pred, LABELS))

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)
error_table(te_x, te_y, pred, LABELS, k=3)

## Results table so far

In [ ]:
# Copy the MLflow database back to Drive (run this after every training cell you care about)
shutil.copy("/content/mlflow.db", f"{DRIVE}/mlruns/mlflow.db")
!python -m src.results_table --out {DRIVE}/results.md